# Aula 14 — Trabalho prático (entrega no Moodle)
## Quem vai ficar insatisfeito? Classificação dos pedidos da hamburgueria

**Análise e Ciência de Dados · Especialização em Ciência de Dados · IFRN**

Você vai aplicar o **fluxo de trabalho do scikit-learn** (Aula 14) ao dataset de delivery da Aula 12 (`dados/delivery_hamburgueria.csv`, 600 pedidos **fictícios**).

**Pergunta:** a partir das informações do pedido e da entrega, o modelo consegue identificar os clientes **insatisfeitos** (avaliação **≤ 3**)?

**Como entregar:** complete as células marcadas com `TODO`, **execute o notebook inteiro** (as saídas devem ficar visíveis) e envie o arquivo `.ipynb` na tarefa *Trabalho Prático — Aula 14* do Moodle. **Prazo: 23/11/2026, 23:59.**

### Critérios de avaliação (100 pontos)
| Etapa | Pontos |
|---|---|
| 1. Alvo e proporção das classes | 10 |
| 2. X, y e divisão treino/teste estratificada | 15 |
| 3. Baseline (`DummyClassifier`) | 10 |
| 4. Pipeline + regressão logística, com métricas no teste (acurácia, F1, ROC-AUC, matriz de confusão) | 25 |
| 5. Segundo modelo no mesmo pipeline | 10 |
| 6. Validação cruzada estratificada (5 folds) comparando os modelos | 20 |
| 7. Conclusão escrita | 10 |

> Use `random_state=42` em todos os lugares para que os resultados possam ser conferidos.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score,
                             ConfusionMatrixDisplay, classification_report)
SEMENTE = 42

df = pd.read_csv("dados/delivery_hamburgueria.csv")
df.head()

## Etapa 1 — Alvo e proporção das classes (10 pts)

Crie a coluna `insatisfeito` = 1 quando `avaliacao <= 3` e 0 caso contrário. Mostre a contagem e a proporção de cada classe. As classes estão balanceadas?

> ✅ Para conferir: **167 pedidos insatisfeitos (27,8%)**.

In [ ]:
# TODO: df["insatisfeito"] = ...
# TODO: mostre value_counts() e value_counts(normalize=True)

## Etapa 2 — X, y e divisão treino/teste (15 pts)

Use como atributos as colunas numéricas `distancia_km`, `itens`, `valor_pedido`, `taxa_entrega`, `tempo_entrega_min` e as categóricas `dia_semana`, `turno`, `bairro`, `veiculo`, `chuva`. **Não use `avaliacao`** (por quê? responda em um comentário). Separe 25% para teste, **estratificando** pelo alvo, com `random_state=42`.

> ✅ Treino com 450 pedidos e teste com 150, ambos com cerca de 28% de insatisfeitos.

In [ ]:
numericas = ["distancia_km", "itens", "valor_pedido", "taxa_entrega", "tempo_entrega_min"]
categoricas = ["dia_semana", "turno", "bairro", "veiculo", "chuva"]
# TODO: X = ...   y = ...
# TODO: X_treino, X_teste, y_treino, y_teste = train_test_split(...)
# Por que não usar "avaliacao"? -> TODO (comentário)

## Etapa 3 — Baseline (10 pts)

Treine um `DummyClassifier(strategy="most_frequent")` e calcule a acurácia e o F1 no teste. O que esse resultado mostra?

> ✅ Acurácia do baseline ≈ **0,72** e F1 = 0.

In [ ]:
# TODO: dummy = DummyClassifier(...).fit(...)
# TODO: acurácia e F1 no teste

## Etapa 4 — Pipeline com regressão logística (25 pts)

Monte um `ColumnTransformer` (`StandardScaler` nas numéricas, `OneHotEncoder(handle_unknown="ignore")` nas categóricas) e um `Pipeline` com `LogisticRegression(max_iter=1000)`. Treine no treino e, **no teste**, mostre acurácia, F1, ROC-AUC (use `predict_proba`), a matriz de confusão e o `classification_report`.

> ✅ Seu modelo deve superar o baseline em F1 e ROC-AUC (referência: ROC-AUC entre 0,75 e 0,85).

In [ ]:
# TODO: preprocess = ColumnTransformer([...])
# TODO: modelo_log = Pipeline([("preprocess", preprocess), ("clf", LogisticRegression(max_iter=1000))])
# TODO: fit, predict, predict_proba
# TODO: métricas + ConfusionMatrixDisplay.from_predictions(...) + classification_report(...)

## Etapa 5 — Segundo modelo (10 pts)

Troque apenas o classificador do pipeline por uma `DecisionTreeClassifier(max_depth=4, random_state=42)` **ou** uma `RandomForestClassifier(n_estimators=300, min_samples_leaf=5, random_state=42)` e calcule as mesmas métricas no teste.

> ✅ Compare com a regressão logística: qual se saiu melhor em cada métrica?

In [ ]:
# TODO: modelo_2 = Pipeline([("preprocess", preprocess), ("clf", ...)])
# TODO: métricas no teste

## Etapa 6 — Validação cruzada (20 pts)

Com `StratifiedKFold(n_splits=5, shuffle=True, random_state=42)` e `cross_validate` **no conjunto de treino**, compare baseline, regressão logística e o seu segundo modelo usando `scoring=["accuracy", "f1", "roc_auc"]`. Monte uma tabela com a **média** e o **desvio-padrão** de cada métrica.

> ✅ Uma tabela com 3 linhas (modelos) e as métricas em média ± desvio-padrão.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEMENTE)
# TODO: para cada modelo, cross_validate(pipeline, X_treino, y_treino, cv=cv, scoring=[...])
# TODO: monte um DataFrame com média e desvio-padrão

## Etapa 7 — Conclusão (10 pts)
Em uma célula Markdown (5 a 8 linhas), responda:
1. Qual modelo você escolheria e com base em qual métrica? Por que **não** usar só a acurácia?
2. O que a matriz de confusão diz sobre os erros do modelo (falsos positivos × falsos negativos)?
3. A variável `tempo_entrega_min` só é conhecida **depois** da entrega. Se a empresa quisesse prever a insatisfação **no momento do pedido**, o que mudaria no seu modelo?

**TODO — escreva aqui a sua conclusão.**